# Predicting Brain Tumor Volume Using MRI-Derived Features and Machine Learning

This notebook analyzes a participant-level MRI dataset and evaluates whether MRI-derived features can predict brain tumor volume (`tumor_size_ml`).

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

## 2. Load Final Modeling Dataset

This file should be the merged participant-level dataset containing demographics, MRI-derived features, QC metrics, and tumor volume.

In [ ]:
df = pd.read_csv("final_tumor_size_modeling_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

## 3. Basic Cleaning and Target Definition

In [ ]:
# Remove rows with missing target values.
df = df.dropna(subset=["tumor_size_ml"]).copy()

# Remove unused error/message columns if present.
df = df.drop(columns=["Error Message"], errors="ignore")

# Target variable
y = df["tumor_size_ml"]

print("Cleaned dataset shape:", df.shape)
print(y.describe())

## 4. Feature Set

Tumor-related label columns are excluded from the predictors to avoid data leakage.

In [ ]:
features = [
    "age",
    "has_motor_scan",
    "has_language_scan",
    "mean_tSNR",
    "mean_FD_mm",
    "n_brain_voxels",
    "n_scrubbed_volumes",
    "n_volumes_after_scrub",
    "tr_seconds",
    "Mean Intensity",
    "Median Intensity",
    "Std Intensity",
    "Nonzero Voxels",
    "Nonzero Mean Intensity",
    "Approx Brain Voxels",
    "Approx Brain Volume ml",
    "Approx Brain Mean Intensity",
    "Simple SNR",
    "Image Volume ml"
]

# Keep only features that exist in the dataframe.
features = [col for col in features if col in df.columns]

X = df[features]
y = df["tumor_size_ml"]

print("Number of features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:")
print(X.isnull().sum()[X.isnull().sum() > 0])

## 5. Dataset Overview and Missing Values

In [ ]:
print(df.info())

In [ ]:
missing = df.isnull().sum()
missing[missing > 0]

## 6. Summary Statistics

In [ ]:
summary_stats = df[["tumor_size_ml"] + features].describe()
summary_stats

## 7. Tumor Size Distribution

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["tumor_size_ml"], bins=15, edgecolor="black")
plt.title("Distribution of Tumor Size")
plt.xlabel("Tumor Size (ml)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.boxplot(df["tumor_size_ml"])
plt.title("Boxplot of Tumor Size")
plt.ylabel("Tumor Size (ml)")
plt.show()

## 8. Correlation Analysis

In [ ]:
corr = df[["tumor_size_ml"] + features].corr(numeric_only=True)

corr_target = corr["tumor_size_ml"].sort_values(ascending=False)
corr_target

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

## 9. Scatterplots of Selected MRI Features vs Tumor Size

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df["Nonzero Voxels"], df["tumor_size_ml"])
plt.xlabel("Nonzero Voxels")
plt.ylabel("Tumor Size (ml)")
plt.title("Nonzero Voxels vs Tumor Size")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df["Approx Brain Voxels"], df["tumor_size_ml"])
plt.xlabel("Approx Brain Voxels")
plt.ylabel("Tumor Size (ml)")
plt.title("Approx Brain Voxels vs Tumor Size")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df["Simple SNR"], df["tumor_size_ml"])
plt.xlabel("Simple SNR")
plt.ylabel("Tumor Size (ml)")
plt.title("Simple SNR vs Tumor Size")
plt.show()

## 10. Multicollinearity Check Using VIF

VIF is calculated using the final predictor set. Very high VIF values suggest redundant predictors.

In [ ]:
X_vif = X.copy()

# Remove constant columns if any exist.
X_vif = X_vif.loc[:, X_vif.nunique() > 1]

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

vif_data.sort_values("VIF", ascending=False)

## 11. Train/Test Split

The same split is used for all regression models for fair comparison.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 12. LASSO Feature Selection

In [ ]:
lasso_fs = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=5, max_iter=100000, random_state=42))
])

lasso_fs.fit(X, y)

coef = pd.Series(
    lasso_fs.named_steps["lasso"].coef_,
    index=X.columns
)

selected = coef[coef != 0].sort_values(key=abs, ascending=False)
selected

## 13. Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
r2_lr = r2_score(y_test, pred_lr)

print("Linear Regression")
print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print("R2:", r2_lr)

## 14. Ridge Regression

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]

ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=alphas))
])

ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)

mae_ridge = mean_absolute_error(y_test, pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_ridge))
r2_ridge = r2_score(y_test, pred_ridge)

print("Ridge Regression")
print("Best Alpha:", ridge.named_steps["ridge"].alpha_)
print("MAE:", mae_ridge)
print("RMSE:", rmse_ridge)
print("R2:", r2_ridge)

## 15. LASSO Regression Model

This uses LASSO as a prediction model, not only as feature selection.

In [ ]:
lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=5, max_iter=100000, random_state=42))
])

lasso_model.fit(X_train, y_train)
pred_lasso = lasso_model.predict(X_test)

mae_lasso = mean_absolute_error(y_test, pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, pred_lasso))
r2_lasso = r2_score(y_test, pred_lasso)

print("LASSO Regression")
print("MAE:", mae_lasso)
print("RMSE:", rmse_lasso)
print("R2:", r2_lasso)

## 16. Decision Tree Regression

In [ ]:
tree = DecisionTreeRegressor(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

pred_tree = tree.predict(X_test)

mae_tree = mean_absolute_error(y_test, pred_tree)
rmse_tree = np.sqrt(mean_squared_error(y_test, pred_tree))
r2_tree = r2_score(y_test, pred_tree)

print("Decision Tree")
print("MAE:", mae_tree)
print("RMSE:", rmse_tree)
print("R2:", r2_tree)

In [ ]:
tree_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
}).sort_values("Importance", ascending=False)

tree_importance.head(10)

In [ ]:
plt.figure(figsize=(15,8))
plot_tree(
    tree,
    feature_names=X.columns,
    filled=True,
    fontsize=8
)
plt.title("Decision Tree Regressor")
plt.show()

## 17. Random Forest Regression

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
r2_rf = r2_score(y_test, pred_rf)

print("Random Forest")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2:", r2_rf)

In [ ]:
rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

rf_importance.head(10)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=rf_importance.head(10),
    x="Importance",
    y="Feature"
)
plt.title("Top 10 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 18. Tuned Random Forest Regression

In [ ]:
rf_tuned = RandomForestRegressor(
    n_estimators=1000,
    max_depth=3,
    min_samples_leaf=2,
    random_state=42
)

rf_tuned.fit(X_train, y_train)
pred_rf_tuned = rf_tuned.predict(X_test)

mae_rf_tuned = mean_absolute_error(y_test, pred_rf_tuned)
rmse_rf_tuned = np.sqrt(mean_squared_error(y_test, pred_rf_tuned))
r2_rf_tuned = r2_score(y_test, pred_rf_tuned)

print("Tuned Random Forest")
print("MAE:", mae_rf_tuned)
print("RMSE:", rmse_rf_tuned)
print("R2:", r2_rf_tuned)

## 19. Model Comparison

In [ ]:
model_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "LASSO Regression",
        "Decision Tree",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "MAE": [
        mae_lr,
        mae_ridge,
        mae_lasso,
        mae_tree,
        mae_rf,
        mae_rf_tuned
    ],
    "RMSE": [
        rmse_lr,
        rmse_ridge,
        rmse_lasso,
        rmse_tree,
        rmse_rf,
        rmse_rf_tuned
    ],
    "R2": [
        r2_lr,
        r2_ridge,
        r2_lasso,
        r2_tree,
        r2_rf,
        r2_rf_tuned
    ]
})

model_results.sort_values("RMSE")

## 20. PCA Dimension Reduction

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

pca = PCA()
pca.fit(X_scaled)

explained = pca.explained_variance_ratio_
cum_explained = explained.cumsum()

plt.figure(figsize=(8,5))
plt.plot(
    range(1, len(cum_explained) + 1),
    cum_explained,
    marker="o"
)
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.grid(True)
plt.show()

In [ ]:
pca_table = pd.DataFrame({
    "PC": range(1, len(explained) + 1),
    "Variance Explained": explained,
    "Cumulative Variance": cum_explained
})

pca_table.head(10)

## 21. Classification: Small vs Large Tumor

A binary classification target is created using the median tumor size.

In [ ]:
median_size = df["tumor_size_ml"].median()

df["large_tumor"] = (df["tumor_size_ml"] > median_size).astype(int)

print("Median tumor size:", median_size)
print(df["large_tumor"].value_counts())

In [ ]:
X_class = X
y_class = df["large_tumor"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class,
    y_class,
    test_size=0.25,
    random_state=42
)

logit = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=5000))
])

logit.fit(X_train_c, y_train_c)
pred_class = logit.predict(X_test_c)

classification_accuracy = accuracy_score(y_test_c, pred_class)
classification_cm = confusion_matrix(y_test_c, pred_class)

print("Accuracy:", classification_accuracy)
print("Confusion Matrix:")
print(classification_cm)

## 22. Generalized Linear Model (Gaussian GLM)

In [ ]:
X_glm = sm.add_constant(X)

glm = sm.GLM(
    y,
    X_glm,
    family=sm.families.Gaussian()
)

glm_results = glm.fit()
print(glm_results.summary())

## 23. Optional Log-Transformed Target Experiment

Because tumor size is right-skewed, a log-transformed target is tested as an additional experiment. This section should be discussed only if it improves model performance.

In [ ]:
df["log_tumor_size"] = np.log1p(df["tumor_size_ml"])
y_log = df["log_tumor_size"]

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X,
    y_log,
    test_size=0.25,
    random_state=42
)

rf_log = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)

rf_log.fit(X_train_log, y_train_log)
pred_rf_log = rf_log.predict(X_test_log)

r2_rf_log = r2_score(y_test_log, pred_rf_log)
rmse_rf_log = np.sqrt(mean_squared_error(y_test_log, pred_rf_log))

print("Random Forest with Log Target")
print("R2:", r2_rf_log)
print("RMSE:", rmse_rf_log)

## 24. Final Summary

This notebook compares linear, regularized, tree-based, dimensionality reduction, classification, and GLM approaches for predicting or analyzing brain tumor volume from MRI-derived features.